┌─────────────────┐      ┌─────────────────┐      ┌─────────────────┐      ┌─────────────────┐
  │ 1. Ingestion    │ ───► │ 2. Telemetry    │ ───► │ 3. Relational   │ ───► │ 4. Analytics    │
  │    (REST API)   │      │    Cleaning     │      │    Warehouse    │      │    Dashboard    │
  └─────────────────┘      └─────────────────┘      └─────────────────┘      └─────────────────┘

In [30]:
import pandas as pd
import os
import requests
import numpy as np

In [ ]:
baseline_url = "https://opensky-network.org/api/states/all?lamin=15.0&lamax=32.0&lomin=34.0&lomax=60.0"
response = requests.get(baseline_url)
print(response.status_code)

200


In [5]:
# 1. Store the JSON response
json_data = response.json()

# 2. Define the exact column schema according to OpenSky API specs
columns = [
    "icao24", "callsign", "origin_country", "time_position", 
    "last_contact", "longitude", "latitude", "baro_altitude", 
    "on_ground", "velocity", "true_track", "vertical_rate", 
    "sensors", "geo_altitude", "squawk", "spi", "position_source"
]

# 3. Create a clean, flat DataFrame
df_flights = pd.DataFrame(json_data['states'], columns=columns)

# 4. Add the snapshot timestamp as a metadata column
df_flights['snapshot_time'] = json_data['time']

# Preview the clean tabular data
df_flights.head()

,icao24,callsign,origin_country,time_position,last_contact,longitude,latitude,baro_altitude,on_ground,velocity,true_track,vertical_rate,sensors,geo_altitude,squawk,spi,position_source,snapshot_time
0,80163d,IGO098,India,1784020980,1784021010,53.0533,22.7560,9448.80,False,230.42,97.57,-0.33,None,10119.36,3113,False,0,1784021019
1,801650,AXB426,India,1784020454,1784021012,55.7091,24.9064,7330.44,False,200.56,120.18,4.23,None,NaN,6210,False,0,1784021019
2,800448,AXB748,India,1784020674,1784021018,55.6733,24.9240,7505.70,False,206.89,120.65,8.45,None,5105.40,6214,False,0,1784021019
3,80044b,AXB411,India,1784021019,1784021019,55.6517,25.2538,762.00,False,99.47,302.54,1.95,None,647.70,4716,False,0,1784021019
4,896024,FVS442,United Arab Emirates,1784021006,1784021015,52.9713,24.5207,3489.96,False,124.54,95.45,0.00,None,3657.60,0437,False,0,1784021019


In [6]:

# Clean up callsign whitespace
df_flights['callsign'] = df_flights['callsign'].str.strip()

# Convert to numeric and convert meters to feet
df_flights['altitude_ft'] = pd.to_numeric(df_flights['baro_altitude'], errors='coerce') * 3.28084 

# Convert Velocity from m/s to knots
df_flights['velocity_knots'] = pd.to_numeric(df_flights['velocity'], errors='coerce') * 1.94384

# Convert Vertical Rate from m/s to feet/min
df_flights['vertical_rate_ftp'] = pd.to_numeric(df_flights['vertical_rate'], errors='coerce') * 196.850394

# Convert Unix timestamps to UTC standard datetime
df_flights['time_position'] = pd.to_datetime(df_flights['time_position'], unit='s', utc=True)
df_flights['snapshot_time'] = pd.to_datetime(df_flights['snapshot_time'], unit='s', utc=True)

df_flights.head()


,icao24,callsign,origin_country,time_position,last_contact,longitude,latitude,baro_altitude,on_ground,velocity,...,vertical_rate,sensors,geo_altitude,squawk,spi,position_source,snapshot_time,altitude_ft,velocity_knots,vertical_rate_ftp
0,80163d,IGO098,India,2026-07-14 09:23:00+00:00,1784021010,53.0533,22.7560,9448.80,False,230.42,...,-0.33,None,10119.36,3113,False,0,2026-07-14 09:23:39+00:00,31000.000992,447.899613,-64.960630
1,801650,AXB426,India,2026-07-14 09:14:14+00:00,1784021012,55.7091,24.9064,7330.44,False,200.56,...,4.23,None,NaN,6210,False,0,2026-07-14 09:23:39+00:00,24050.000770,389.856550,832.677167
2,800448,AXB748,India,2026-07-14 09:17:54+00:00,1784021018,55.6733,24.9240,7505.70,False,206.89,...,8.45,None,5105.40,6214,False,0,2026-07-14 09:23:39+00:00,24625.000788,402.161058,1663.385829
3,80044b,AXB411,India,2026-07-14 09:23:39+00:00,1784021019,55.6517,25.2538,762.00,False,99.47,...,1.95,None,647.70,4716,False,0,2026-07-14 09:23:39+00:00,2500.000080,193.353765,383.858268
4,896024,FVS442,United Arab Emirates,2026-07-14 09:23:26+00:00,1784021015,52.9713,24.5207,3489.96,False,124.54,...,0.00,None,3657.60,0437,False,0,2026-07-14 09:23:39+00:00,11450.000366,242.085834,0.000000


In [11]:
df_flights['flight_status'] = np.select(
    [df_flights['on_ground'] == True , df_flights['vertical_rate_ftp'] > 300, df_flights['vertical_rate_ftp'] < -300],
    ['Ground', 'Climbing', 'Descending'],
    default='Cruising'
)

df_flights['flight_status'].value_counts()

flight_status
Cruising      63
Climbing      32
Descending    22
Ground         3
Name: count, dtype: int64

In [12]:
df_flights.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 22 columns):
 #   Column             Non-Null Count  Dtype              
---  ------             --------------  -----              
 0   icao24             120 non-null    object             
 1   callsign           120 non-null    object             
 2   origin_country     120 non-null    object             
 3   time_position      120 non-null    datetime64[ns, UTC]
 4   last_contact       120 non-null    int64              
 5   longitude          120 non-null    float64            
 6   latitude           120 non-null    float64            
 7   baro_altitude      118 non-null    float64            
 8   on_ground          120 non-null    bool               
 9   velocity           120 non-null    float64            
 10  true_track         120 non-null    float64            
 11  vertical_rate      117 non-null    float64            
 12  sensors            0 non-null      object         

In [14]:
# Hub Airport Coordinates (Latitude, Longitude)
AIRPORTS = {
    'DXB': (25.2532, 55.3657),  # Dubai International
    'AUH': (24.4330, 54.6511),  # Abu Dhabi International
    'DOH': (25.2731, 51.6081),  # Hamad International (Doha)
    'RUH': (24.9576, 46.6988)   # King Khalid International (Riyadh)
}

def haversine_km(lat1, lon1, lat2, lon2):
    # Convert degrees to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    km = 6371.0 * c  # Earth radius in kilometers
    return km

# 1. Compute distance (in km) to each hub
for hub_code, (hub_lat, hub_lon) in AIRPORTS.items():
    df_flights[f'dist_{hub_code}_km'] = haversine_km(
        df_flights['latitude'], df_flights['longitude'], 
        hub_lat, hub_lon
    )

# 2. Extract column names created for distances
hub_dist_cols = [f'dist_{code}_km' for code in AIRPORTS.keys()]

# 3. Find the minimum distance and assign the corresponding airport code
df_flights['dist_to_hub_km'] = df_flights[hub_dist_cols].min(axis=1)
df_flights['nearest_hub'] = df_flights[hub_dist_cols].idxmin(axis=1).str.replace('dist_', '').str.replace('_km', '')

df_flights[['dist_to_hub_km', 'nearest_hub'] + hub_dist_cols].head()

,dist_to_hub_km,nearest_hub,dist_DXB_km,dist_AUH_km,dist_DOH_km,dist_RUH_km
0,247.543562,AUH,363.677863,247.543562,316.036468,690.923077
1,51.798867,DXB,51.798867,119.163005,414.974491,908.382442
2,47.953058,DXB,47.953058,116.823862,411.168762,904.699804
3,28.762499,DXB,28.762499,136.101726,406.613431,901.901667
4,160.945671,DOH,254.880386,170.277016,160.945671,635.260423


In [15]:
df_flights.drop(columns=hub_dist_cols, inplace=True)

In [20]:
df_flights.drop(columns=['last_contact'], inplace=True)
df_flights.rename(columns={'vertical_rate_ftp': 'vertical_rate_fpm'}, inplace=True)

In [34]:
df_flights.drop(columns=['time_position'], inplace=True)
df_flights.head()

,icao24,callsign,origin_country,longitude,latitude,on_ground,snapshot_time,altitude_ft,velocity_knots,vertical_rate_fpm,flight_status,dist_to_hub_km,nearest_hub
0,80163d,IGO098,India,53.0533,22.7560,False,2026-07-14 09:23:39+00:00,31000.000992,447.899613,-64.960630,Cruising,247.543562,AUH
1,801650,AXB426,India,55.7091,24.9064,False,2026-07-14 09:23:39+00:00,24050.000770,389.856550,832.677167,Climbing,51.798867,DXB
2,800448,AXB748,India,55.6733,24.9240,False,2026-07-14 09:23:39+00:00,24625.000788,402.161058,1663.385829,Climbing,47.953058,DXB
3,80044b,AXB411,India,55.6517,25.2538,False,2026-07-14 09:23:39+00:00,2500.000080,193.353765,383.858268,Climbing,28.762499,DXB
4,896024,FVS442,United Arab Emirates,52.9713,24.5207,False,2026-07-14 09:23:39+00:00,11450.000366,242.085834,0.000000,Cruising,160.945671,DOH


In [35]:
os.makedirs('../data/processed', exist_ok=True)
df_flights.to_csv('../data/processed/flights_snapshot.csv', index=False)
print("CSV file created successfully")

CSV file created successfully


In [36]:
from sqlalchemy import create_engine

# Update with your local postgres username, password, host, and port
DB_USER = "postgres"
DB_PASS = "Regal%40273"  # Replace with your actual password
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "gulf_aviation"

# Create Database Engine
engine = create_engine(f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

# Append clean snapshot into PostgreSQL
df_flights.to_sql('fact_flight_states', con=engine, if_exists='append', index=False)
print("Flight snapshot successfully written to PostgreSQL!")

Flight snapshot successfully written to PostgreSQL!
